In [1]:
import sys
import os

# Set the main path in the root folder of the project.
sys.path.append(os.path.join('..'))

In [2]:
# Settings for autoreloading.
%load_ext autoreload
%autoreload 2

In [3]:
from src.utils.seed import set_random_seed

# Set the random seed for deterministic operations.
SEED = 42
set_random_seed(SEED)

In [4]:
import os

BASE_DATA_DIR = os.path.join('..', 'data', 'metr-la')

In [5]:
from src.data.data_extraction import get_adjacency_matrix

# Get the adjacency matrix
adj_matrix_structure = get_adjacency_matrix(
    os.path.join(BASE_DATA_DIR, 'raw', 'adj_mx_metr_la.pkl'))

# Get the header of the adjacency matrix, the node indices and the
# matrix itself.
header, node_ids_dict, adj_matrix = adj_matrix_structure

In [6]:
from src.data.data_extraction import get_locations_dataframe

# Get the dataframe containing the latitude and longitude of each sensor.
locations_df = get_locations_dataframe(
    os.path.join(BASE_DATA_DIR, 'raw', 'graph_sensor_locations_metr_la.csv'),
    has_header=True)

In [7]:
# Get the node positions dictionary.
node_pos_dict = { i: id for id, i in node_ids_dict.items() }

In [8]:
import os
import numpy as np
from src.spatial_temporal_gnn.prediction import predict

# Get the data and the values predicted by the STGNN.
x_train = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_train.npy'))[..., :1]
x_val = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_val.npy'))[..., :1]
x_test = np.load(os.path.join(BASE_DATA_DIR, 'explained', 'x_test.npy'))[..., :1]

In [9]:
from src.utils.config import MPH_TO_KMH_FACTOR

# Turn the dataset in kilometers per hour.
x_train = x_train * MPH_TO_KMH_FACTOR
x_val = x_val * MPH_TO_KMH_FACTOR
x_test = x_test * MPH_TO_KMH_FACTOR

In [10]:
_, n_timesteps, n_nodes, _ = x_train.shape

In [12]:
from src.explanation.clustering.clustering import (
    get_adjacency_distance_matrix)

adj_distance_matrix = get_adjacency_distance_matrix(adj_matrix, n_timesteps)

In [14]:
print(f'Shape of the Adjacency Distance Matrix: {adj_distance_matrix.shape}')

Shape of the Adjacency Distance Matrix: (2484, 2484)


In [15]:
from src.explanation.clustering.clustering import (
    get_temporal_distance_matrix)

temporal_distance_matrix = get_temporal_distance_matrix(n_nodes, n_timesteps)

In [16]:
print('Shape of the Temporal Distance Matrix:',
      f'{temporal_distance_matrix.shape}')

Shape of the Temporal Distance Matrix: (2484, 2484)


In [160]:
DATA_DIR = os.path.join(BASE_DATA_DIR, 'clustered')

os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
from src.explanation.clustering.clustering_explanations import get_explanation_clusters_dataset

x_train_clustered = get_explanation_clusters_dataset(
    x_train,
    adj_distance_matrix,
    temporal_distance_matrix)

np.save(os.path.join(DATA_DIR, 'x_train.npy'), x_train_clustered)

x_val_clustered = get_explanation_clusters_dataset(
    x_val,
    adj_distance_matrix,
    temporal_distance_matrix)

np.save(os.path.join(DATA_DIR, 'x_val.npy'), x_val_clustered)

x_test_clustered = get_explanation_clusters_dataset(
    x_test,
    adj_distance_matrix,
    temporal_distance_matrix)

np.save(os.path.join(DATA_DIR, 'x_test.npy'), x_test_clustered)